# U5-04 · Validación, métricas y clases desequilibradas

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 5, apartados 5.7 a 5.9*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

## 1. Validación cruzada anidada (ejemplo 5.8)

Son 16 combinaciones × 5 particiones internas × 5 externas: en el navegador puede tardar un poco.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

Xc, yc = load_breast_cancer(return_X_y=True)
svm = make_pipeline(StandardScaler(), SVC())
rejilla = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}
interna = StratifiedKFold(5, shuffle=True, random_state=1)
externa = StratifiedKFold(5, shuffle=True, random_state=2)
busqueda = GridSearchCV(svm, rejilla, cv=interna).fit(Xc, yc)
print("Mejor combinación:", busqueda.best_params_, "· su puntuación:", round(busqueda.best_score_, 3))
print("Validación anidada:", round(cross_val_score(busqueda, Xc, yc, cv=externa).mean(), 3))

## 2. Piezas defectuosas: un problema desequilibrado (ejemplo 5.10)

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef

X, y = make_classification(n_samples=5000, n_features=12, n_informative=5, n_redundant=2,
                           weights=[0.96], flip_y=0.01, random_state=7)
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print("Defectuosas:", y.sum(), "de", len(y), "· en prueba:", y_pru.sum(), "de", len(y_pru))
base = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent)
print("Exactitud de decir siempre «correcta»:", round(base.score(X_pru, y_pru), 3))

In [ ]:
logit = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_ent, y_ent)
pred = logit.predict(X_pru)
print(confusion_matrix(y_pru, pred))
print(classification_report(y_pru, pred, digits=3))
print("MCC:", round(matthews_corrcoef(y_pru, pred), 3))

## 3. Las métricas a mano (ejemplo 5.9)

Comprueba que coinciden con las de scikit-learn.

In [ ]:
vn, fp, fn, vp = confusion_matrix(y_pru, pred).ravel()
exactitud = (vp + vn) / (vp + fn + fp + vn)
precision = vp / (vp + fp)
exhaustividad = vp / (vp + fn)
f1 = 2 * precision * exhaustividad / (precision + exhaustividad)
mcc = (vp * vn - fp * fn) / np.sqrt((vp + fp) * (vp + fn) * (vn + fp) * (vn + fn))
print(f"exactitud {exactitud:.3f} · precisión {precision:.3f} · exhaustividad {exhaustividad:.3f} · F1 {f1:.3f} · MCC {mcc:.3f}")

## 4. Curvas ROC y de precisión-exhaustividad (figura 5.8)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_estimator(logit, X_pru, y_pru, ax=a1, plot_chance_level=True)
PrecisionRecallDisplay.from_estimator(logit, X_pru, y_pru, ax=a2, plot_chance_level=True)
plt.tight_layout(); plt.show()

## 5. Pesos por clase (ejemplo 5.11)

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
pesos = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(X_ent, y_ent)
for nombre, m in {"sin pesos": logit, "con pesos": pesos}.items():
    p = m.predict(X_pru)
    print(f"{nombre:>9}: precisión {precision_score(y_pru, p):.3f} · exhaustividad {recall_score(y_pru, p):.3f}"
          f" · F1 {f1_score(y_pru, p):.3f} · MCC {matthews_corrcoef(y_pru, p):.3f}"
          f" · AUC {roc_auc_score(y_pru, m.predict_proba(X_pru)[:, 1]):.3f}")
print(confusion_matrix(y_pru, pesos.predict(X_pru)))

## 6. Elegir el umbral sin mirar los datos de prueba

El umbral es un hiperparámetro más: se elige con validación cruzada en los datos de entrenamiento. `TunedThresholdClassifierCV` lo hace por nosotros.

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV
ajustado = TunedThresholdClassifierCV(logit, scoring="f1", cv=5).fit(X_ent, y_ent)
p = ajustado.predict(X_pru)
print("Umbral elegido:", round(ajustado.best_threshold_, 3))
print(confusion_matrix(y_pru, p), "\nF1 en prueba:", round(f1_score(y_pru, p), 3))

**Ejercicio.** Repite el apartado 6 con `scoring="recall"`. ¿Qué umbral elige y por qué no es una buena idea optimizar solo la exhaustividad?

> Saito, T. y Rehmsmeier, M. (2015). *PLOS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432
>
> Varma, S. y Simon, R. (2006). Bias in Error Estimation When Using Cross-Validation for Model Selection. *BMC Bioinformatics*, 7, 91. https://doi.org/10.1186/1471-2105-7-91